# Frequentist vs Bayesian Inference in a Marketing Experiment

### In this simulation I want to simulate two mostly used methods in statistics named Frequentist and Baysian to see how these two work and estimate the value for parameters  better say for treatment effect. 

## Scenario

We imagine a scenario in which streaming platform wants to test whether a personalized discount email increases subscription purchases.

Users are randomly assigned to:

Control: standard email
Treatment: personalized discount email

Outcome:

$$
Y_i =
\begin{cases}
1 & \text{if user subscribes} \
0 & \text{otherwise}
\end{cases}
$$

The company wants to compare:

Frequentist inference
Bayesian inference

for estimating the treatment effect.

## Part 1 — Data Generating Process


In part 1 we will simulate or generate the data for the variables. 

Assuming there are (N = 20000) users.

Each user has covariates:

$$
X_i = \text{prior engagement score}
$$

$$
Z_i = \text{income segment}
$$

Treatment assignment:

$$
D_i \sim \text{Bernoulli}(0.5)
$$

True purchase probability:

$$
P(Y_i = 1) = \text{logit}^{-1}(\alpha + \tau D_i + \beta X_i + \gamma Z_i)
$$

where:

$$
\alpha = -3
$$

$$
\tau = 0.35
$$

$$
\beta = 0.8
$$

$$
\gamma = 0.5
$$

Tasks would be: 

Simulating (X_i \sim N(0,1))
Simulating (Z_i \sim \text{Bernoulli}(0.4))
Randomly assigning treatment (D_i)
Generating outcome (Y_i)
Calculating the true average treatment effect:

$$
ATE = E[Y_i(1) - Y_i(0)]
$$

## Part 2 — Frequentist Analysis

In this part we will estimate the treatment effect using:

### Model 1: Difference in Means

$$
\hat{\tau} = \bar{Y}_T - \bar{Y}_C
$$

Tasks would be:

Estimating treatment effect
Computing standard error
Constructing 95% confidence interval
Conducting hypothesis test:

$$
H_0: \tau = 0
$$

$$
H_1: \tau \neq 0
$$

### Model 2: Logistic Regression

Estimating:

$$
\text{logit}(P(Y_i = 1)) = \alpha + \tau D_i + \beta X_i + \gamma Z_i
$$

Tasks:

Estimating (\hat{\tau})
Converting log-odds effect into marginal effect
Reporting standard errors
Reporting p-value
Comparing with true ATE

## Part 3 — Frequentist Challenges

We will discuss these problems step by step:

### Challenge 1: P-value dependence on sample size

With large (N), small effects may become statistically significant.

Question would be : Is the result practically meaningful or only statistically significant?


### Challenge 2: Sequential testing

We suppose team checks results every 1,000 users and stops when (p < 0.05).

Question: Why does this inflate false positives?

We will simulate repeated looks and compare:

Fixed-sample test
Sequential test

### Challenge 4: Heterogeneous treatment effects

We will estimate effects separately for:

High engagement users
Low engagement users
High income users
Low income users

Question: Are subgroup effects reliable, or are they noisy?



## Part 4 — Bayesian Analysis

Use a Bayesian logistic regression:

**𝑌ᵢ ∼ Bernoulli(𝑝ᵢ)**

**logit(𝑝ᵢ) = 𝛼 + 𝜏𝐷ᵢ + 𝛽𝑋ᵢ + 𝛾𝑍ᵢ**

### Set Priors

**𝛼 ∼ 𝑁(0,5)**

**𝜏 ∼ 𝑁(0,1)**

**𝛽 ∼ 𝑁(0,1)**

**𝛾 ∼ 𝑁(0,1)**

## Tasks

- Estimate posterior distribution of 𝜏
- Compute posterior mean
- Compute 95% credible interval
- Compute:

  **𝑃(𝜏 > 0 ∣ 𝑑𝑎𝑡𝑎)**

- Compute:

  **𝑃(𝐴𝑇𝐸 > 0.01 ∣ 𝑑𝑎𝑡𝑎)**

## Part 5 — Bayesian Challenges

### Challenge 1: Prior Sensitivity

Repeat the model using:

### Weak Prior

**𝜏 ∼ 𝑁(0,5)**

### Skeptical Prior

**𝜏 ∼ 𝑁(0,0.2)**

### Optimistic Prior

**𝜏 ∼ 𝑁(0.5,0.5)**

### Question

How much does the posterior change?

### Challenge 2: Small Sample Problem

Repeat the simulation with:

**𝑁 = 200**

### Question

Does the prior matter more when sample size is small?

In [3]:
import numpy as np
import pandas as pd
import statsmodels.api as sm

In [1]:
import numpy as np
import pandas as pd

np.random.seed(42)

#Number of customers
N = 200000

# Engagement score
# Around -3 to +3, average around 0
X = np.random.normal(0, 1, N)

# Income segment
# 1 = high income, 0 = low income
Z = np.random.binomial(1, 0.4, N)

# Treatment assignment
# 1 = personalized email, 0 = standard email
D = np.random.binomial(1, 0.5, N)

#  True coefficients

alpha = -3     # baseline purchase tendency
tau = 0.35     # treatment effect
beta = 0.8     # engagement effect
gamma = 0.5    # income effect


# Purchase probability

linear_score = alpha + tau*D + beta*X + gamma*Z

p = 1 / (1 + np.exp(-linear_score))

#  purchase outcome
Y = np.random.binomial(1, p)


# dataframe

df = pd.DataFrame({
    "customer_id": np.arange(1, N+1),
    "engagement_X": X,
    "income_Z": Z,
    "treatment_D": D,
    "purchase_probability": p,
    "purchase_Y": Y
})

df.head()

,customer_id,engagement_X,income_Z,treatment_D,purchase_probability,purchase_Y
0,1,0.496714,0,0,0.068969,1
1,2,-0.138264,1,1,0.094438,0
2,3,0.647689,1,0,0.121122,0
3,4,1.523030,1,1,0.282605,1
4,5,-0.234153,0,0,0.039646,0


In [2]:
df.groupby("treatment_D")["purchase_Y"].mean()

treatment_D
0    0.075300
1    0.102197
Name: purchase_Y, dtype: float64

In [4]:
treated_mean = df[df["treatment_D"] == 1]["purchase_Y"].mean()
control_mean = df[df["treatment_D"] == 0]["purchase_Y"].mean()

tau_hat = treated_mean - control_mean

print("Treatment rate:", treated_mean)
print("Control rate:", control_mean)
print("ATE estimate:", tau_hat)

Treatment rate: 0.10219745031720935
Control rate: 0.07530030180081146
ATE estimate: 0.026897148516397892


$$
ATE
=
\overline{Y}_{D=1}
-
\overline{Y}_{D=0}
$$

$$
ATE
=
0.1022
-
0.0753
=
0.0269
$$

The purchase rate was 10.22% for the treatment group and 7.53% for the control group. Therefore, the personalized email increased the purchase probability by approximately **2.69 percentage points on average**.

In [5]:
treated = df[df["treatment_D"] == 1]["purchase_Y"]
control = df[df["treatment_D"] == 0]["purchase_Y"]

se = np.sqrt(
    treated.var(ddof=1)/len(treated)
    +
    control.var(ddof=1)/len(control)
)

print("Standard Error:", se)

Standard Error: 0.0012704313575343893


The standard error of the difference in purchase rates is:

$$
SE(ATE)
=
\sqrt{
\frac{s_1^2}{n_1}
+
\frac{s_0^2}{n_0}
}
$$

$$
SE(ATE)=0.00127
$$

This means the estimated treatment effect of 0.0269 has relatively little sampling uncertainty. Across repeated samples, the estimated ATE would typically vary by about 0.00127, or 0.127 percentage points.

In [6]:
lower = tau_hat - 1.96 * se
upper = tau_hat + 1.96 * se

print("95% CI:")
print(lower, upper)

95% CI:
0.02440710305563049 0.029387193977165296


**95% CI:**

$$
95\%CI
=
ATE
\pm
1.96
\times
SE(ATE)
$$

$$
0.0269
\pm
1.96(0.00127)
=
[0.0244,0.0294]
$$

We are 95% confident that the personalized email increases the purchase probability by approximately 2.44 to 2.94 percentage points. Since the interval does not include zero, the treatment effect is statistically significant at the 5% level.

In [7]:
from scipy import stats

t_stat, p_value = stats.ttest_ind(
    treated,
    control,
    equal_var=False
)

print("t-statistic:", t_stat)
print("p-value:", p_value)

t-statistic: 21.17166610922347
p-value: 2.2529461052648298e-99


**The hypothesis test is:**

$$
H_0:\mu_1-\mu_0=0
$$

$$
H_1:\mu_1-\mu_0\neq0
$$

**The test statistic is approximately:**

$$
t
=
\frac{\overline{Y}_1-\overline{Y}_0}
{SE(\overline{Y}_1-\overline{Y}_0)}
=
\frac{0.0269}{0.00127}
\approx21.17
$$

Since:

$$
p\approx2.25\times10^{-99}
$$

the p-value is extremely small. We reject the null hypothesis and conclude that the personalized email has a statistically significant effect on the purchase rate. The estimated increase is about 2.69 percentage points.

In [8]:
if p_value < 0.05:
    print("Reject H0")
else:
    print("Fail to reject H0")

Reject H0


In [9]:
import statsmodels.api as sm

X_reg = sm.add_constant(df["treatment_D"])

model = sm.OLS(
    df["purchase_Y"],
    X_reg
).fit()

print(model.summary())

                            OLS Regression Results                            
Dep. Variable:             purchase_Y   R-squared:                       0.002
Model:                            OLS   Adj. R-squared:                  0.002
Method:                 Least Squares   F-statistic:                     448.3
Date:                Tue, 28 Jul 2026   Prob (F-statistic):           2.15e-99
Time:                        16:22:42   Log-Likelihood:                -32067.
No. Observations:              200000   AIC:                         6.414e+04
Df Residuals:                  199998   BIC:                         6.416e+04
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                  coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------
const           0.0753      0.001     83.858      

**The estimated OLS model is:**

$$
\widehat{\text{purchase}_Y}
=
0.0753
+
0.0269\,\text{treatment}_D
$$

## Interpretation

- **Intercept = 0.0753:** Customers receiving the standard email have an estimated purchase rate of 7.53%.

- **Treatment coefficient = 0.0269:** The personalized email increases the purchase rate by about 2.69 percentage points.

Therefore, the predicted purchase rate for treated customers is:

$$
0.0753+0.0269=0.1022
$$

or approximately 10.22%.

## Statistical Significance

The treatment coefficient has:

$$
t=21.17,\qquad p<0.001
$$

The 95% confidence interval is:

$$
[0.024,0.029]
$$

Because the p-value is extremely small and the confidence interval does not include zero, the treatment effect is statistically significant.

## R-squared

$$
R^2=0.002
$$

Treatment alone explains only about 0.2% of the individual variation in purchasing. This is not surprising because purchase behavior also depends on engagement, income, and random factors. A low \(R^2\) does not invalidate the estimated treatment effect.

In [10]:
X_reg = sm.add_constant(
    df[[
        "treatment_D",
        "engagement_X",
        "income_Z"
    ]]
)

model = sm.OLS(
    df["purchase_Y"],
    X_reg
).fit()

print(model.summary())

                            OLS Regression Results                            
Dep. Variable:             purchase_Y   R-squared:                       0.052
Model:                            OLS   Adj. R-squared:                  0.052
Method:                 Least Squares   F-statistic:                     3660.
Date:                Tue, 28 Jul 2026   Prob (F-statistic):               0.00
Time:                        16:31:06   Log-Likelihood:                -26945.
No. Observations:              200000   AIC:                         5.390e+04
Df Residuals:                  199996   BIC:                         5.394e+04
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------
const            0.0595      0.001     58.954   

**The estimated linear probability model is:**

$$
\widehat{\text{purchase}_Y}
=
0.0595
+
0.0264D
+
0.0603X
+
0.0399Z
$$

where \(D\) is treatment, \(X\) is engagement, and \(Z\) is income group.

- **Treatment coefficient = 0.0264:** Holding engagement and income constant, the personalized email increases purchase probability by about 2.64 percentage points.

- **Engagement coefficient = 0.0603:** A one-unit increase in engagement is associated with about a 6.03-percentage-point increase in purchase probability.

- **Income coefficient = 0.0399:** High-income customers have about a 3.99-percentage-point higher purchase probability than low-income customers, holding other variables constant.

- **Intercept = 0.0595:** A low-income customer with engagement \(X=0\) who receives the standard email has a predicted purchase probability of about 5.95%.

All three coefficients are statistically significant because their p-values are below 0.001.

$$
R^2=0.052
$$

This means the model explains about 5.2% of the variation in individual purchase outcomes. The treatment estimate, 2.64 percentage points, is very close to the earlier unadjusted estimate of 2.69 percentage points, which is expected because treatment was randomly assigned.

In [11]:
import statsmodels.api as sm

X_reg = sm.add_constant(
    df[
        ["treatment_D",
         "engagement_X",
         "income_Z"]
    ]
)

logit_model = sm.Logit(
    df["purchase_Y"],
    X_reg
).fit()

print(logit_model.summary())

Optimization terminated successfully.
         Current function value: 0.272798
         Iterations 7
                           Logit Regression Results                           
Dep. Variable:             purchase_Y   No. Observations:               200000
Model:                          Logit   Df Residuals:                   199996
Method:                           MLE   Df Model:                            3
Date:                Tue, 28 Jul 2026   Pseudo R-squ.:                 0.08950
Time:                        16:31:32   Log-Likelihood:                -54560.
converged:                       True   LL-Null:                       -59923.
Covariance Type:            nonrobust   LLR p-value:                     0.000
                   coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------
const           -2.9934      0.016   -190.014      0.000      -3.024      -2.962
treatment_D      0.3494

**The estimated logistic regression is:**

$$
\log\left(\frac{p}{1-p}\right)
=
-2.9934
+
0.3494D
+
0.7975X
+
0.5117Z
$$

where \(p\) is the probability of purchase.

- **Treatment coefficient = 0.3494:** Holding engagement and income constant, the personalized email increases the log-odds of purchasing by 0.3494. In odds terms:

  $$
  e^{0.3494}\approx1.42
  $$

  So, the personalized email increases the odds of purchase by approximately 42%.

- **Engagement coefficient = 0.7975:** A one-unit increase in engagement multiplies the purchase odds by:

  $$
  e^{0.7975}\approx2.22
  $$

  This is about a 122% increase in the odds.

- **Income coefficient = 0.5117:** High-income customers have:

  $$
  e^{0.5117}\approx1.67
  $$

  or approximately 67% higher purchase odds than low-income customers.

All coefficients are statistically significant because their p-values are below 0.001.

The estimates are also very close to the true coefficients used in the data-generating process:

$$
\alpha=-3,\qquad
\tau=0.35,\qquad
\beta=0.8,\qquad
\gamma=0.5
$$

This shows that, with a large sample, logistic regression successfully recovered the true model parameters. The model also converged successfully after seven iterations.

In [4]:
def simulate_customer_data(
    N=200000,
    seed=42,
    alpha=-3,
    tau=0.35,
    beta=0.8,
    gamma=0.5,
    treatment_prob=0.5,
    high_income_prob=0.4,
    engagement_mean=0,
    engagement_sd=1
):


    
    np.random.seed(seed)

    
    X = np.random.normal(engagement_mean, engagement_sd, N)

    
    Z = np.random.binomial(1, high_income_prob, N)

    
    D = np.random.binomial(1, treatment_prob, N)

    
    linear_score = alpha + tau * D + beta * X + gamma * Z

    
    p = 1 / (1 + np.exp(-linear_score))

    
    Y = np.random.binomial(1, p)

    
    df = pd.DataFrame({
        "customer_id": np.arange(1, N + 1),
        "engagement_X": X,
        "income_Z": Z,
        "treatment_D": D,
        "purchase_probability": p,
        "purchase_Y": Y
    })

    return df


df = simulate_customer_data()

df.head()

,customer_id,engagement_X,income_Z,treatment_D,purchase_probability,purchase_Y
0,1,0.496714,0,0,0.068969,1
1,2,-0.138264,1,1,0.094438,0
2,3,0.647689,1,0,0.121122,0
3,4,1.523030,1,1,0.282605,1
4,5,-0.234153,0,0,0.039646,0


In [13]:


sample_sizes = [200, 2000, 20000, 200000]

results = []

for N in sample_sizes:

    
    df = simulate_customer_data(N=N)

    X_reg = sm.add_constant(
        df[["treatment_D", "engagement_X", "income_Z"]]
    )

    # OLS
    ols = sm.OLS(df["purchase_Y"], X_reg).fit()

    # Logistic Regression
    logit = sm.Logit(df["purchase_Y"], X_reg).fit(disp=0)

    
    results.append({
        "N": N,

        "OLS_coef": ols.params["treatment_D"],
        "OLS_SE": ols.bse["treatment_D"],
        "OLS_p": ols.pvalues["treatment_D"],

        "Logit_coef": logit.params["treatment_D"],
        "Logit_SE": logit.bse["treatment_D"],
        "Logit_p": logit.pvalues["treatment_D"]
    })

results = pd.DataFrame(results)

results

,N,OLS_coef,OLS_SE,OLS_p,Logit_coef,Logit_SE,Logit_p
0,200,0.028459,0.037258,4.458940e-01,0.573939,0.584013,3.257303e-01
1,2000,0.018713,0.011927,1.168210e-01,0.301592,0.170076,7.618286e-02
2,20000,0.034231,0.003850,6.527161e-19,0.481210,0.053205,1.503568e-19
3,200000,0.026379,0.001238,1.360413e-100,0.349400,0.016388,7.239245e-101


# Effect of Sample Size on Estimation

This table shows how sample size affects estimation precision and statistical significance.

## OLS Versus Logistic Regression

The OLS model estimates the treatment effect on the purchase-probability scale:

$$
ATE_{\text{OLS}}
\approx
P(Y=1\mid D=1)
-
P(Y=1\mid D=0)
$$

Therefore, its coefficient is around 0.02–0.03, meaning roughly a 2–3 percentage-point increase in purchase probability.

The logistic model estimates the effect on the log-odds scale:

$$
\log\left(\frac{p}{1-p}\right)
=
\alpha+\tau D+\beta X+\gamma Z
$$

The true treatment coefficient used to generate the data was:

$$
\tau=0.35
$$

At \(N=200{,}000\), the estimated logit coefficient is:

$$
\widehat{\tau}=0.3494
$$

which is almost exactly the true value.

## Effect of Sample Size

| Sample size | Interpretation |
|---:|---|
| 200 | Estimates are noisy, standard errors are large, and neither model finds a significant effect. |
| 2,000 | Estimates become more precise, but the p-values are still above 0.05. |
| 20,000 | Standard errors are much smaller, and the effect becomes strongly significant. |
| 200,000 | Estimates are highly precise; logistic regression closely recovers the true coefficient of 0.35. |

As \(N\) increases:

$$
SE\downarrow,
\qquad
|t|\text{ or }|z|\uparrow,
\qquad
p\text{-value}\downarrow
$$

The small-sample insignificant results do not mean the treatment has no effect. They mean that the samples of 200 and 2,000 customers do not provide enough precision to clearly distinguish the effect from sampling noise.

In [14]:

df = simulate_customer_data(N=20000, tau=0.35)

looks = range(1000, 20001, 1000)

seq_results = []

for n in looks:
    
    df_temp = df.iloc[:n].copy()  
    X_reg = sm.add_constant(
        df_temp[["treatment_D", "engagement_X", "income_Z"]]
    )
    
    ols = sm.OLS(df_temp["purchase_Y"], X_reg).fit()
    
    seq_results.append({
        "N_checked": n,
        "OLS_coef": ols.params["treatment_D"],
        "OLS_SE": ols.bse["treatment_D"],
        "OLS_p": ols.pvalues["treatment_D"],
        "significant": ols.pvalues["treatment_D"] < 0.05
    })

seq_results = pd.DataFrame(seq_results)

seq_results

,N_checked,OLS_coef,OLS_SE,OLS_p,significant
0,1000,0.039410,0.017191,2.208187e-02,True
1,2000,0.024693,0.011912,3.830533e-02,True
2,3000,0.033537,0.009720,5.673814e-04,True
3,4000,0.029609,0.008608,5.880705e-04,True
4,5000,0.029785,0.007602,9.042943e-05,True
5,6000,0.028021,0.006916,5.147638e-05,True
6,7000,0.029996,0.006346,2.326528e-06,True
7,8000,0.027504,0.005989,4.439947e-06,True
8,9000,0.026568,0.005648,2.591443e-06,True
9,10000,0.030210,0.005358,1.767334e-08,True


# Sequential Testing as Sample Size Grows

This code checks the treatment effect repeatedly as the sample grows from 1,000 to 20,000 customers.

At each sample size, the model is:

$$
Y_i
=
\beta_0
+
\tau D_i
+
\beta_1 X_i
+
\beta_2 Z_i
+
\varepsilon_i
$$

and the hypothesis test is:

$$
H_0:\tau=0
\qquad
\text{vs.}
\qquad
H_1:\tau\neq0
$$

## Interpretation

### At \(N=1000\)

$$
\widehat{\tau}=0.0394,
\qquad
p=0.022
$$

The personalized email is estimated to increase purchase probability by about 3.94 percentage points, and the result is statistically significant.

### At \(N=20000\)

$$
\widehat{\tau}=0.0342,
\qquad
SE=0.00385
$$

The estimated increase is about 3.42 percentage points, with a very small p-value.

As the sample grows:

- The coefficient fluctuates because each new group of observations adds sampling variation.
- The standard error generally decreases.
- The p-value generally becomes smaller.
- The estimate becomes more stable.

## Sequential Testing

We suppose the true treatment effect is zero:

$$
H_0:\tau=0
$$

The team checks the result every 1,000 users and stops as soon as:

$$
p<0.05
$$

For one test, the probability of a false positive is approximately 5%:

$$
P(\text{false positive})=0.05
$$

But with sequential testing, the team gets many opportunities to obtain a significant result by random chance.

For example, the team checks at:

$$
N=1{,}000,\;2{,}000,\;3{,}000,\ldots,20{,}000
$$

That means up to 20 hypothesis tests are performed. Even when the treatment has no real effect, one of the fluctuating estimates may temporarily produce:

$$
p<0.05
$$

The team then stops and incorrectly concludes that the treatment works.

## Fixed-Sample Test

In a fixed-sample design, the team decides in advance to analyze the result only once at \(N=20000\).

$$
P(\text{false positive})\approx5\%
$$

## Sequential Test

In a naive sequential design, the team tests repeatedly and stops after the first significant result.

$$
P(\text{at least one false positive})>5\%
$$

If the 20 tests were independent, the approximate probability would be:

$$
1-(1-0.05)^{20}
=
1-0.95^{20}
\approx0.642
$$

So the false-positive probability could be about 64%. In this example, the tests are not independent because each larger sample contains the earlier observations, so the exact rate will differ. However, it will still be higher than 5%.

Therefore, this simulation compare:

- **Fixed-sample false-positive rate**

with:

- **Sequential-testing false-positive rate**

under a data-generating process where:

$$
\tau=0
$$

The fixed test should produce false positives in about 5% of simulations, while the naive sequential test should produce false positives more often.

In [15]:
df = simulate_customer_data(N=20000, tau=0)

looks = range(1000, 20001, 1000)

seq_results = []

for n in looks:
    
    df_temp = df.iloc[:n].copy()
    
    X_reg = sm.add_constant(
        df_temp[["treatment_D", "engagement_X", "income_Z"]]
    )
    
    ols = sm.OLS(df_temp["purchase_Y"], X_reg).fit()
    
    seq_results.append({
        "N_checked": n,
        "OLS_coef": ols.params["treatment_D"],
        "OLS_p": ols.pvalues["treatment_D"],
        "significant": ols.pvalues["treatment_D"] < 0.05
    })

seq_results = pd.DataFrame(seq_results)

seq_results

,N_checked,OLS_coef,OLS_p,significant
0,1000,0.016629,0.305803,False
1,2000,-0.000571,0.959101,False
2,3000,0.009693,0.289085,False
3,4000,0.003170,0.693179,False
4,5000,0.004397,0.535339,False
5,6000,0.002036,0.751863,False
6,7000,0.004190,0.476745,False
7,8000,0.001446,0.794771,False
8,9000,0.000685,0.896198,False
9,10000,0.004301,0.387534,False


# Sequential Testing When the True Treatment Effect Is Zero

This code performs sequential testing when the true treatment effect is zero:

$$
\tau=0
$$

At each sample size, the model tests:

$$
H_0:\tau=0
\qquad
\text{vs.}
\qquad
H_1:\tau\neq0
$$

## Interpretation

From \(N=1000\) to \(N=16000\), all p-values are above 0.05, so the model correctly finds no statistically significant treatment effect.

At \(N=17000\):

$$
\widehat{\tau}=0.00786,
\qquad
p=0.0413
$$

The model temporarily suggests that treatment increases purchase probability by about 0.79 percentage points, even though the true treatment effect is exactly zero.

At \(N=18000\):

$$
\widehat{\tau}=0.00740,
\qquad
p=0.0482
$$

The result remains temporarily significant. But after adding more observations:

$$
p_{19000}=0.0535
$$

and:

$$
p_{20000}=0.0719
$$

The result is no longer statistically significant. This shows that p-values can move above and below 0.05 as new random observations are added.

## Fixed-Sample Versus Sequential Testing

### Fixed-Sample Test

We suppose the team decides in advance to test only once at:

$$
N=20000
$$

The final result is:

$$
\widehat{\tau}=0.00643,
\qquad
p=0.0719
$$

Because \(p>0.05\), the team does not reject the null hypothesis. This is the correct conclusion because the true effect is zero.

### Sequential Test

We suppose the team checks every 1,000 users and stops at the first result where:

$$
p<0.05
$$

The team would stop at \(N=17000\), because:

$$
p=0.0413
$$

It would incorrectly conclude that treatment has an effect. This is a false positive.

The problem is that every additional check gives random sampling variation another opportunity to produce \(p<0.05\). Therefore:

$$
P(\text{at least one false positive across repeated looks})>0.05
$$


In [16]:
seq_results["significant"].any()

np.True_

In [19]:
def simulate_customer_data_hte(
    N=20000,
    seed=42,
    alpha=-3,
    tau=0.35,
    beta=0.8,
    gamma=0.5,
    delta=0.4
):
    np.random.seed(seed)

    X = np.random.normal(0, 1, N)
    Z = np.random.binomial(1, 0.4, N)
    D = np.random.binomial(1, 0.5, N)

    # True heterogeneous treatment effect:
    # treatment effect = tau + delta * X
    linear_score = alpha + tau*D + beta*X + gamma*Z + delta*(D*X)

    p = 1 / (1 + np.exp(-linear_score))
    Y = np.random.binomial(1, p)

    df = pd.DataFrame({
        "customer_id": np.arange(1, N + 1),
        "engagement_X": X,
        "income_Z": Z,
        "treatment_D": D,
        "purchase_probability": p,
        "purchase_Y": Y
    })

    return df

df=simulate_customer_data_hte()
df.head()

,customer_id,engagement_X,income_Z,treatment_D,purchase_probability,purchase_Y
0,1,0.496714,0,0,0.068969,0
1,2,-0.138264,1,0,0.068459,0
2,3,0.647689,1,1,0.202172,0
3,4,1.523030,1,1,0.420100,0
4,5,-0.234153,1,1,0.080840,0


In [23]:

df['engagment_treatment']=df['treatment_D']*df['engagement_X']

x_reg=sm.add_constant(df[['engagement_X','income_Z','treatment_D','engagment_treatment']])

model=sm.OLS(df['purchase_Y'], x_reg
).fit()

print(model.summary())

                            OLS Regression Results                            
Dep. Variable:             purchase_Y   R-squared:                       0.102
Model:                            OLS   Adj. R-squared:                  0.102
Method:                 Least Squares   F-statistic:                     565.8
Date:                Tue, 28 Jul 2026   Prob (F-statistic):               0.00
Time:                        18:47:44   Log-Likelihood:                -3033.5
No. Observations:               20000   AIC:                             6077.
Df Residuals:                   19995   BIC:                             6117.
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                          coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------
const                   0.0511    

**The estimated OLS interaction model is:**

$$
Y
=
0.0511
+
0.0527X
+
0.0435Z
+
0.0577D
+
0.0600(D\times X)
$$

**The estimated treatment effect depends on engagement:**

$$
TE(X)
=
0.0577
+
0.0600X
$$

## Interpretation

- **Treatment coefficient = 0.0577:** For a customer with average engagement, \(X=0\), the personalized email increases purchase probability by about 5.77 percentage points.

- **Interaction coefficient = 0.0600:** For every one-unit increase in engagement, the treatment effect increases by another 6 percentage points.

For example:

### \(X=-1\)

$$
TE(-1)
=
0.0577
+
0.0600(-1)
=
-0.0023
$$

The estimated effect is approximately zero for a low-engagement customer.

### \(X=0\)

$$
TE(0)
=
0.0577
$$

The estimated effect is 5.77 percentage points for an average-engagement customer.

### \(X=1\)

$$
TE(1)
=
0.0577
+
0.0600(1)
=
0.1177
$$

The estimated effect is 11.77 percentage points for a higher-engagement customer.

Therefore, the personalized email is more effective for customers with higher engagement. The interaction is statistically significant because:

$$
p<0.001
$$

One important distinction is that the true values \(\tau=0.35\) and \(\delta=0.4\) were defined on the log-odds scale in the logistic data-generating process. The OLS estimates \(0.0577\) and \(0.0600\) are effects on the approximate purchase-probability scale, so they should not be expected to equal \(0.35\) and \(0.4\).

In [24]:
df = simulate_customer_data_hte(N=20000)

median_engagement = df["engagement_X"].median()
df["high_engagement"] = (df["engagement_X"] >= median_engagement).astype(int)

def estimate_subgroup_effect(data, group_name):

    X_reg = sm.add_constant(
        data[["treatment_D", "engagement_X", "income_Z"]]
    )

    model = sm.OLS(data["purchase_Y"], X_reg).fit()

    return {
        "group": group_name,
        "N": len(data),
        "ATE_estimate": model.params["treatment_D"],
        "SE": model.bse["treatment_D"],
        "p_value": model.pvalues["treatment_D"],
        "CI_lower": model.conf_int().loc["treatment_D", 0],
        "CI_upper": model.conf_int().loc["treatment_D", 1]
    }

results = []

results.append(
    estimate_subgroup_effect(
        df[df["high_engagement"] == 1],
        "High engagement"
    )
)

results.append(
    estimate_subgroup_effect(
        df[df["high_engagement"] == 0],
        "Low engagement"
    )
)

results.append(
    estimate_subgroup_effect(
        df[df["income_Z"] == 1],
        "High income"
    )
)

results.append(
    estimate_subgroup_effect(
        df[df["income_Z"] == 0],
        "Low income"
    )
)

hte_results = pd.DataFrame(results)

hte_results


,group,N,ATE_estimate,SE,p_value,CI_lower,CI_upper
0,High engagement,10000,0.103371,0.006966,2.734846e-49,0.089716,0.117026
1,Low engagement,10000,0.014940,0.003730,6.250764e-05,0.007627,0.022252
2,High income,7937,0.066388,0.007045,5.609642e-21,0.052578,0.080198
3,Low income,12063,0.052548,0.004748,2.441439e-28,0.043242,0.061854


# Subgroup Treatment-Effect Analysis

This table estimates the treatment effect separately inside different subgroups.

For each subgroup, the model is:

$$
Y_i
=
\beta_0
+
\tau D_i
+
\beta_1X_i
+
\beta_2Z_i
+
\varepsilon_i
$$

Here, \(\tau\) is estimated using only the observations belonging to that subgroup.

## Interpretation of the Results

### Engagement Subgroups

For high-engagement customers:

$$
ATE=0.1034
$$

The personalized email increases purchase probability by about 10.34 percentage points.

For low-engagement customers:

$$
ATE=0.0149
$$

The personalized email increases purchase probability by about 1.49 percentage points.

The treatment effect is therefore much larger among high-engagement customers:

$$
0.1034-0.0149=0.0885
$$

So the estimated difference between the two subgroup effects is about 8.85 percentage points.

This is consistent with the data-generating process because it included:

$$
\delta(D\times X)
$$

meaning that the treatment effect becomes larger as engagement increases.

### Income Subgroups

For high-income customers:

$$
ATE=0.0664
$$

For low-income customers:

$$
ATE=0.0525
$$

The estimated effect is about 1.39 percentage points larger for high-income customers:

$$
0.0664-0.0525=0.0139
$$

However, the data-generating process did not include a treatment-by-income interaction. Therefore, this difference should not automatically be interpreted as true income-based heterogeneity. It may arise from sampling variation, differences in engagement composition, or the nonlinear relationship between log-odds and probability.

## Difference From the Previous Interaction Analysis

### Previous Pooled Interaction Model

Previously, we used all 20,000 observations in one model:

$$
Y
=
\beta_0
+
\beta_1X
+
\beta_2Z
+
\beta_3D
+
\beta_4(D\times X)
+
\varepsilon
$$

The treatment effect was:

$$
TE(X)=0.0577+0.0600X
$$

This approach treats engagement as a continuous variable. It estimates how the treatment effect changes for every one-unit increase in engagement.

**Advantages:**

- It uses the full dataset in one regression.
- It keeps all information in the continuous engagement variable.
- The interaction coefficient directly tests whether the treatment effect changes with engagement.
- It provides a smooth treatment-effect function rather than only two categories.

### Current Subgroup Analysis

In the current analysis, we first divide customers into groups, such as high and low engagement, and then fit separate regressions:

- High-engagement sample
- Low-engagement sample

Each regression still uses all variables available within that subgroup, but it only uses part of the full dataset.

This produces one average treatment effect for each group:

$$
ATE_{\text{high}}
$$

and:

$$
ATE_{\text{low}}
$$

The subgroup approach is easier to explain, but it has some disadvantages:

- Each model uses fewer observations.
- Dividing engagement at the median loses information.
- Customers just above and below the median are treated as belonging to completely different groups.
- It does not directly test whether the two subgroup effects are statistically different.

A significant p-value in both groups means:

$$
ATE_{\text{high}}\neq0
$$

and:

$$
ATE_{\text{low}}\neq0
$$

It does not by itself prove:

$$
ATE_{\text{high}}\neq ATE_{\text{low}}
$$

For that formal comparison, the pooled interaction model is better.



So we can say the subgroup table is useful for presenting the results in an intuitive way:

- **High engagement:** approximately 10.34 percentage-point effect
- **Low engagement:** approximately 1.49 percentage-point effect

But the interaction model is statistically stronger for testing heterogeneity because it uses all observations together and directly estimates whether the treatment effect changes with engagement. The best practice is to use the interaction model for formal testing and the subgroup table for clear interpretation.

In [5]:
import pymc as pm


df_bayes = simulate_customer_data(N=500, tau=0.35)

D = df_bayes["treatment_D"].values
X = df_bayes["engagement_X"].values
Z = df_bayes["income_Z"].values
Y = df_bayes["purchase_Y"].values

with pm.Model() as bayes_logit:

    alpha = pm.Normal("alpha", mu=0, sigma=5)
    tau = pm.Normal("tau", mu=0, sigma=1)
    beta = pm.Normal("beta", mu=0, sigma=1)
    gamma = pm.Normal("gamma", mu=0, sigma=1)

    linear_score = alpha + tau*D + beta*X + gamma*Z
    p = pm.math.sigmoid(linear_score)

    Y_obs = pm.Bernoulli("Y_obs", p=p, observed=Y)

    trace = pm.sample(
        draws=500,
        tune=500,
        chains=1,
        cores=1,
        random_seed=42
    )

Initializing NUTS using jitter+adapt_diag...
Sequential sampling (1 chains in 1 job)
NUTS: [alpha, tau, beta, gamma]


Output()

Sampling 1 chain for 500 tune and 500 draw iterations (500 + 500 draws total) took 2 seconds.
Only one chain was sampled, this makes it impossible to run some convergence checks


# Bayesian Logistic Model

The Bayesian logistic model is:

$$
\operatorname{logit}(p_i)
=
\alpha
+
\tau D_i
+
\beta X_i
+
\gamma Z_i
$$

or equivalently:

$$
p_i
=
\frac{1}{
1+\exp\left[
-\left(
\alpha
+
\tau D_i
+
\beta X_i
+
\gamma Z_i
\right)
\right]
}
$$

The observed purchase outcome is modeled as:

$$
Y_i
\sim
\operatorname{Bernoulli}(p_i)
$$

## Priors

$$
\alpha\sim N(0,5)
$$

$$
\tau,\beta,\gamma\sim N(0,1)
$$

These priors describe the plausible parameter values before observing the simulated data.



In [7]:
# mean, sd, 95% credible interval

import arviz as az

az.summary(
    trace,
    var_names=["alpha", "tau", "beta", "gamma"],
    hdi_prob=0.95
)

arviz - WARNING - Shape validation failed: input_shape: (1, 500), minimum_shape: (chains=2, draws=4)


,mean,sd,hdi_2.5%,hdi_97.5%,mcse_mean,mcse_sd,ess_bulk,ess_tail,r_hat
alpha,-2.979,0.310,-3.587,-2.405,0.019,0.020,275.0,293.0,NaN
tau,0.371,0.346,-0.285,0.997,0.023,0.016,247.0,260.0,NaN
beta,0.832,0.156,0.502,1.125,0.007,0.008,494.0,226.0,NaN
gamma,0.445,0.306,-0.149,1.033,0.016,0.014,379.0,320.0,NaN


# Posterior Summary

We interpert posterior summary like following: 

## Bayesian Treatment Effect

$$
\tau\mid Y
\approx
N\left(0.371,\;0.346^2\right)
$$

The posterior mean is:

$$
E(\tau\mid Y)=0.371
$$

This is close to the true value used in the simulation:

$$
\tau_{\text{true}}=0.35
$$

However, the 95% credible interval is:

$$
[-0.285,\;0.997]
$$

Because this interval includes zero, the data do not provide strong evidence that the treatment effect is positive. The posterior estimate is centered near the true value, but it is highly uncertain because the sample contains only 500 customers.

## Other Coefficients

- **Alpha:** Posterior mean \(=-2.979\), close to the true value \(-3\). Its credible interval excludes zero.

- **Beta:** Posterior mean \(=0.832\), close to the true engagement coefficient \(0.8\). Its credible interval:

  $$
  [0.502,\;1.125]
  $$

  excludes zero, indicating strong evidence of a positive engagement effect.

- **Gamma:** Posterior mean \(=0.445\), close to the true income coefficient \(0.5\). However, its interval:

  $$
  [-0.149,\;1.033]
  $$

  includes zero, so the income effect remains uncertain.

## Main Conclusion

The Bayesian model recovered posterior means reasonably close to the true parameters:

$$
\alpha=-3,\qquad
\tau=0.35,\qquad
\beta=0.8,\qquad
\gamma=0.5
$$

But with only \(N=500\), the posterior distributions for treatment and income are wide. Therefore, close posterior means do not necessarily mean that the effects have been estimated precisely.

for the following sections we increase the chain and also other parameters. 

In [8]:
# Posterior Mean of Tau

tau_samples = trace.posterior["tau"].values.flatten()

posterior_mean_tau = tau_samples.mean()

posterior_mean_tau

np.float64(0.3705658131298563)

# Posterior Mean

$$
E(\tau\mid Y)=0.371
$$

The estimated treatment effect is 0.371 on the log-odds scale, close to the true simulated value:

$$
\tau_{\text{true}}=0.35
$$

In odds terms:

$$
e^{0.371}\approx1.45
$$

So the posterior mean suggests that the personalized email increases purchase odds by approximately 45%, holding engagement and income constant.

In [9]:
# 95% Credible Interval for Tau


credible_interval_tau = np.percentile(tau_samples, [2.5, 97.5])

credible_interval_tau

array([-0.28209268,  0.99494356])

# 95% Credible Interval

$$
95\%\,CI_{\tau}
=
[-0.282,\;0.995]
$$

This means that, given the model, priors, and observed data, approximately 95% of the sampled posterior values of \(\tau\) fall between \(-0.282\) and \(0.995\).

Because the interval includes zero, there is still considerable uncertainty about whether the treatment effect is positive.

In [10]:
# Probability Tau Is Positive


prob_tau_positive = np.mean(tau_samples > 0)

prob_tau_positive

np.float64(0.814)

# Probability That the Treatment Effect Is Positive

$$
P(\tau>0\mid Y)=0.814
$$

This means that 81.4% of the posterior samples indicate a positive treatment effect. Equivalently, given this model and data, there is an estimated 81.4% posterior probability that the personalized email increases purchase odds.

However, 81.4% is not especially strong evidence by common Bayesian decision thresholds such as 95%. The result leans toward a positive effect, but the sample of 500 customers does not provide enough information for a confident conclusion.

The small difference between our percentile interval:

$$
[-0.282,\;0.995]
$$

and ArviZ’s HDI:

$$
[-0.285,\;0.997]
$$

is normal because they use slightly different methods to construct the credible interval.

In [11]:
# Compute Bayesian ATE

# For each posterior sample, we compute:

# p1 = probability if everyone gets treatment
# p0 = probability if nobody gets treatment
# ATE = average(p1 - p0)



alpha_samples = trace.posterior["alpha"].values.flatten()
tau_samples = trace.posterior["tau"].values.flatten()
beta_samples = trace.posterior["beta"].values.flatten()
gamma_samples = trace.posterior["gamma"].values.flatten()

ate_samples = []

for a, t, b, g in zip(alpha_samples, tau_samples, beta_samples, gamma_samples):

    # everyone control
    p0 = 1 / (1 + np.exp(-(a + b*X + g*Z)))

    # everyone treatment
    p1 = 1 / (1 + np.exp(-(a + t + b*X + g*Z)))

    ate = np.mean(p1 - p0)

    ate_samples.append(ate)

ate_samples = np.array(ate_samples)

# Bayesian ATE on the Purchase-Probability Scale

This code converts the Bayesian logistic-regression coefficients into a treatment effect on the purchase-probability scale.

For every posterior draw, it calculates:

$$
p_{0i}
=
\operatorname{logit}^{-1}
\left(
\alpha+\beta X_i+\gamma Z_i
\right)
$$

$$
p_{1i}
=
\operatorname{logit}^{-1}
\left(
\alpha+\tau+\beta X_i+\gamma Z_i
\right)
$$

Then:

$$
ATE
=
\frac{1}{N}
\sum_{i=1}^{N}
\left(
p_{1i}-p_{0i}
\right)
$$

This compares each customer under two hypothetical situations:

- The customer receives the standard email.
- The same customer receives the personalized email.

The differences are then averaged across all 500 customers.






In [12]:
# Posterior Mean of ATE

ate_samples.mean()

np.float64(0.027803245929591262)

## Posterior Mean of the ATE

$$
E(ATE\mid Y)=0.0278
$$

The personalized email is estimated to increase the average purchase probability by approximately 2.78 percentage points.

This is different from the posterior mean of:

$$
\tau=0.371
$$

because \(\tau\) is measured on the log-odds scale, while the Bayesian ATE is measured directly on the probability scale.

In [13]:
# 95% Credible Interval for ATE

np.percentile(ate_samples, [2.5, 97.5])

array([-0.02181739,  0.07428282])

## 95% Credible Interval

$$
95\%\,CI_{ATE}
=
[-0.0218,\;0.0743]
$$

The plausible average treatment effects range from approximately:

- A 2.18-percentage-point decrease
- To a 7.43-percentage-point increase

Because the credible interval includes zero, there is still substantial uncertainty about whether the treatment improves purchase probability.

In [14]:
# Probability ATE > 0.01

prob_ate_above_001 = np.mean(ate_samples > 0.01)

prob_ate_above_001

np.float64(0.734)

## Probability That the ATE Exceeds One Percentage Point

$$
P(ATE>0.01\mid Y)=0.734
$$

There is a 73.4% posterior probability that the personalized email increases purchase probability by more than 1 percentage point.

This is a useful Bayesian business interpretation. However, 73.4% is not very strong evidence. The results suggest a potentially useful positive effect, but the sample of 500 customers does not provide enough certainty to confidently conclude that the effect exceeds one percentage point.

In [22]:
# Prior Sensitivity Analysis

# Same data for all priors
df_bayes = simulate_customer_data(N=200, tau=0.35)

D = df_bayes["treatment_D"].values
X = df_bayes["engagement_X"].values
Z = df_bayes["income_Z"].values
Y = df_bayes["purchase_Y"].values

In [23]:
priors = {
    "baseline_prior_N_0_1": {"mu": 0, "sigma": 1},
    "weak_prior_N_0_5": {"mu": 0, "sigma": 5},
    "skeptical_prior_N_0_0.2": {"mu": 0, "sigma": 0.2},
    "optimistic_prior_N_0.5_0.5": {"mu": 0.5, "sigma": 0.5}
}

all_results = []
all_traces = {}

for prior_name, prior_values in priors.items():

    with pm.Model() as bayes_logit:

        alpha = pm.Normal("alpha", mu=0, sigma=5)
        tau = pm.Normal(
            "tau",
            mu=prior_values["mu"],
            sigma=prior_values["sigma"]
        )
        beta = pm.Normal("beta", mu=0, sigma=1)
        gamma = pm.Normal("gamma", mu=0, sigma=1)

        linear_score = alpha + tau*D + beta*X + gamma*Z
        p = pm.math.sigmoid(linear_score)

        Y_obs = pm.Bernoulli("Y_obs", p=p, observed=Y)

        trace = pm.sample(
            draws=100,
            tune=100,
            chains=3,
            cores=3,
            random_seed=42,
            progressbar=True
        )

    all_traces[prior_name] = trace

    tau_samples = trace.posterior["tau"].values.flatten()

    alpha_samples = trace.posterior["alpha"].values.flatten()
    beta_samples = trace.posterior["beta"].values.flatten()
    gamma_samples = trace.posterior["gamma"].values.flatten()

    ate_samples = []

    for a, t, b, g in zip(alpha_samples, tau_samples, beta_samples, gamma_samples):

        p0 = 1 / (1 + np.exp(-(a + b*X + g*Z)))
        p1 = 1 / (1 + np.exp(-(a + t + b*X + g*Z)))

        ate_samples.append(np.mean(p1 - p0))

    ate_samples = np.array(ate_samples)

    all_results.append({
        "prior": prior_name,

        "tau_mean": tau_samples.mean(),
        "tau_CI_lower": np.percentile(tau_samples, 2.5),
        "tau_CI_upper": np.percentile(tau_samples, 97.5),
        "P_tau_positive": np.mean(tau_samples > 0),

        "ATE_mean": ate_samples.mean(),
        "ATE_CI_lower": np.percentile(ate_samples, 2.5),
        "ATE_CI_upper": np.percentile(ate_samples, 97.5),
        "P_ATE_above_0.01": np.mean(ate_samples > 0.01)
    })

prior_sensitivity_results = pd.DataFrame(all_results)

prior_sensitivity_results

Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (3 chains in 3 jobs)
NUTS: [alpha, tau, beta, gamma]


Output()

Sampling 3 chains for 100 tune and 100 draw iterations (300 + 300 draws total) took 44 seconds.
We recommend running at least 4 chains for robust computation of convergence diagnostics
The rhat statistic is larger than 1.01 for some parameters. This indicates problems during sampling. See https://arxiv.org/abs/1903.08008 for details
The effective sample size per chain is smaller than 100 for some parameters.  A higher number is needed for reliable rhat and ess computation. See https://arxiv.org/abs/1903.08008 for details
Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (3 chains in 3 jobs)
NUTS: [alpha, tau, beta, gamma]


Output()

Sampling 3 chains for 100 tune and 100 draw iterations (300 + 300 draws total) took 31 seconds.
We recommend running at least 4 chains for robust computation of convergence diagnostics
The rhat statistic is larger than 1.01 for some parameters. This indicates problems during sampling. See https://arxiv.org/abs/1903.08008 for details
The effective sample size per chain is smaller than 100 for some parameters.  A higher number is needed for reliable rhat and ess computation. See https://arxiv.org/abs/1903.08008 for details
Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (3 chains in 3 jobs)
NUTS: [alpha, tau, beta, gamma]


Output()

Sampling 3 chains for 100 tune and 100 draw iterations (300 + 300 draws total) took 28 seconds.
We recommend running at least 4 chains for robust computation of convergence diagnostics
The rhat statistic is larger than 1.01 for some parameters. This indicates problems during sampling. See https://arxiv.org/abs/1903.08008 for details
The effective sample size per chain is smaller than 100 for some parameters.  A higher number is needed for reliable rhat and ess computation. See https://arxiv.org/abs/1903.08008 for details
Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (3 chains in 3 jobs)
NUTS: [alpha, tau, beta, gamma]


Output()

Sampling 3 chains for 100 tune and 100 draw iterations (300 + 300 draws total) took 22 seconds.
We recommend running at least 4 chains for robust computation of convergence diagnostics
The rhat statistic is larger than 1.01 for some parameters. This indicates problems during sampling. See https://arxiv.org/abs/1903.08008 for details
The effective sample size per chain is smaller than 100 for some parameters.  A higher number is needed for reliable rhat and ess computation. See https://arxiv.org/abs/1903.08008 for details


,prior,tau_mean,tau_CI_lower,tau_CI_upper,P_tau_positive,ATE_mean,ATE_CI_lower,ATE_CI_upper,P_ATE_above_0.01
0,baseline_prior_N_0_1,0.348480,-0.648992,1.320627,0.766667,0.021864,-0.038430,0.085465,0.643333
1,weak_prior_N_0_5,0.537852,-0.636181,1.592057,0.800000,0.033379,-0.041427,0.108622,0.730000
2,skeptical_prior_N_0_0.2,0.064403,-0.313462,0.446113,0.616667,0.004454,-0.018464,0.032120,0.340000
3,optimistic_prior_N_0.5_0.5,0.505527,-0.200905,1.247620,0.910000,0.032733,-0.010925,0.084013,0.840000


# Prior Sensitivity Analysis

This is a prior sensitivity analysis. We fit the same Bayesian logistic model to the same \(N=200\) customers but changed the prior distribution for the treatment coefficient:

$$
\tau
$$

The model is:

$$
\operatorname{logit}(p_i)
=
\alpha
+
\tau D_i
+
\beta X_i
+
\gamma Z_i
$$

Because the sample is small, the data provide limited information about \(\tau\), so the prior has a noticeable influence on the posterior.

## Comparison of the Priors

### Baseline Prior

$$
\tau\sim N(0,1)
$$

Results:

$$
E(\tau\mid Y)=0.348
$$

$$
95\%\,CI=[-0.649,\;1.321]
$$

$$
P(\tau>0\mid Y)=76.7\%
$$

The posterior mean is close to the true simulated value of 0.35, but the interval is wide and includes zero.

On the probability scale:

$$
E(ATE\mid Y)=0.0219
$$

The treatment is estimated to increase purchase probability by about 2.19 percentage points, but there is only a 64.3% probability that the ATE exceeds one percentage point.

---

### Weak Prior

$$
\tau\sim N(0,5)
$$

Results:

$$
E(\tau\mid Y)=0.538
$$

$$
95\%\,CI=[-0.636,\;1.592]
$$

Because this prior allows a very wide range of values, it provides little regularization. The posterior is mainly controlled by the small and noisy dataset.

The estimated ATE is:

$$
E(ATE\mid Y)=0.0334
$$

or about 3.34 percentage points.

The uncertainty is also the largest:

$$
95\%\,CI_{ATE}
=
[-0.0414,\;0.1086]
$$

---

### Skeptical Prior

$$
\tau\sim N(0,0.2)
$$

This prior strongly concentrates treatment effects near zero.

Results:

$$
E(\tau\mid Y)=0.064
$$

$$
95\%\,CI=[-0.313,\;0.446]
$$

The prior pulls the posterior treatment estimate strongly toward zero which is called shrinkage.

The estimated ATE is only:

$$
E(ATE\mid Y)=0.00445
$$

or about 0.45 percentage points.

Furthermore:

$$
P(ATE>0.01\mid Y)=34\%
$$

So we can ay nder this skeptical prior, there is little evidence that the treatment effect exceeds one percentage point.

---

### Optimistic Prior

$$
\tau\sim N(0.5,0.5)
$$

This prior begins with the belief that a positive treatment effect around 0.5 is plausible.

Results:

$$
E(\tau\mid Y)=0.506
$$

$$
P(\tau>0\mid Y)=91\%
$$

The estimated ATE is:

$$
E(ATE\mid Y)=0.0327
$$

or about 3.27 percentage points.

Also:

$$
P(ATE>0.01\mid Y)=84\%
$$

This prior produces the strongest evidence for a practically meaningful positive effect.

## Main Interpretation

The posterior conclusion changes noticeably across priors:

| Prior | Posterior ATE | \(P(ATE>0.01)\) |
|---|---:|---:|
| Baseline | 2.19 percentage points | 64.3% |
| Weak | 3.34 percentage points | 73.0% |
| Skeptical | 0.45 percentage points | 34.0% |
| Optimistic | 3.27 percentage points | 84.0% |

This shows that with only \(N=200\), the data are not strong enough to overcome the prior. The skeptical prior pulls the effect toward zero, while the optimistic prior pulls it toward a larger positive effect.

Therefore, the treatment conclusion is prior-sensitive. Maybe more data would make the likelihood stronger, causing the different posterior estimates to become more similar.

In [18]:
az.summary(
    all_traces["skeptical_prior_N_0_0.2"],
    var_names=["alpha", "tau", "beta", "gamma"],
    hdi_prob=0.95
)

,mean,sd,hdi_2.5%,hdi_97.5%,mcse_mean,mcse_sd,ess_bulk,ess_tail,r_hat
alpha,-3.174,0.449,-4.114,-2.345,0.038,0.034,141.0,163.0,1.00
tau,0.064,0.198,-0.315,0.431,0.011,0.009,315.0,227.0,1.00
beta,1.431,0.356,0.756,2.113,0.026,0.023,183.0,174.0,1.01
gamma,0.264,0.496,-0.671,1.190,0.042,0.032,141.0,155.0,1.01


# Posterior Results Under the Skeptical Prior

This table shows the posterior results under the skeptical prior:

$$
\tau\sim N(0,0.2)
$$

This prior strongly assumes, before seeing the data, that the treatment effect is probably close to zero.

## Treatment Effect \(\tau\)

$$
E(\tau\mid Y)=0.064
$$

$$
95\%\,HDI=[-0.315,\;0.431]
$$

The estimated treatment coefficient is only 0.064 on the log-odds scale, much smaller than the true simulated value:

$$
\tau_{\text{true}}=0.35
$$

The interval includes zero, so the posterior allows both negative and positive treatment effects. We can say here is not strong evidence that the treatment is beneficial.

An especially important comparison is:

$$
\text{Prior SD}=0.20,
\qquad
\text{Posterior SD}=0.198
$$

The posterior uncertainty is almost identical to the prior uncertainty. This indicates that, with only \(N=200\), the data contain little information about \(\tau\), so the skeptical prior dominates and pulls the posterior toward zero.

## Other Parameters

### Intercept

$$
\alpha=-3.174,
\qquad
95\%\,HDI=[-4.114,\;-2.345]
$$

This is reasonably close to the true value:

$$
\alpha_{\text{true}}=-3
$$

### Engagement Effect

$$
\beta=1.431,
\qquad
95\%\,HDI=[0.756,\;2.113]
$$

The interval is entirely positive, so there is strong evidence that engagement increases purchase probability. However, the posterior mean is higher than the true value:

$$
\beta_{\text{true}}=0.8
$$

This difference can occur because the sample is small and contains relatively few purchase events.

### Income Effect

$$
\gamma=0.264,
\qquad
95\%\,HDI=[-0.671,\;1.190]
$$

The interval is wide and includes zero. Therefore, the data do not provide strong evidence about the income effect, even though the true simulated value was 0.5.

## Sampling Diagnostics

The \(\widehat{R}\) values are between 1.00 and 1.01, suggesting that the three chains mixed reasonably well:

$$
\widehat{R}\approx1
$$

The effective sample sizes are moderate, particularly for \(\tau\):

$$
ESS_{\text{bulk}}=315,
\qquad
ESS_{\text{tail}}=227
$$

These diagnostics are much better than having an undefined \(\widehat{R}\) with one chain. However, because we retained only 100 draws per chain, the numerical results are still preliminary.

The main substantive conclusion is that the strong skeptical prior substantially shrinks the treatment estimate toward zero because the small dataset is not informative enough to overcome it.

In [29]:
sample_sizes = [200, 500, 1000,3000,6000,10000,200000]

all_results = []

for N in sample_sizes:

    df_bayes = simulate_customer_data(N=N, tau=0.35)

    D = df_bayes["treatment_D"].values
    X = df_bayes["engagement_X"].values
    Z = df_bayes["income_Z"].values
    Y = df_bayes["purchase_Y"].values

    with pm.Model() as model:

        alpha = pm.Normal("alpha", mu=0, sigma=5)
        tau = pm.Normal("tau", mu=0, sigma=1)
        beta = pm.Normal("beta", mu=0, sigma=1)
        gamma = pm.Normal("gamma", mu=0, sigma=1)

        linear_score = alpha + tau*D + beta*X + gamma*Z
        p = pm.math.sigmoid(linear_score)

        Y_obs = pm.Bernoulli("Y_obs", p=p, observed=Y)

        trace = pm.sample(
            draws=1000,
            tune=1000,
            chains=4,
            cores=4,
            random_seed=42,
            progressbar=True
        )

    tau_samples = trace.posterior["tau"].values.flatten()
    alpha_samples = trace.posterior["alpha"].values.flatten()
    beta_samples = trace.posterior["beta"].values.flatten()
    gamma_samples = trace.posterior["gamma"].values.flatten()

    ate_samples = []

    for a, t, b, g in zip(alpha_samples, tau_samples, beta_samples, gamma_samples):

        p0 = 1 / (1 + np.exp(-(a + b*X + g*Z)))
        p1 = 1 / (1 + np.exp(-(a + t + b*X + g*Z)))

        ate_samples.append(np.mean(p1 - p0))

    ate_samples = np.array(ate_samples)

    all_results.append({
        "N": N,
        "tau_mean": tau_samples.mean(),
        "tau_CI_lower": np.percentile(tau_samples, 2.5),
        "tau_CI_upper": np.percentile(tau_samples, 97.5),
        "P_tau_positive": np.mean(tau_samples > 0),

        "ATE_mean": ate_samples.mean(),
        "ATE_CI_lower": np.percentile(ate_samples, 2.5),
        "ATE_CI_upper": np.percentile(ate_samples, 97.5),
        "P_ATE_above_0.01": np.mean(ate_samples > 0.01)
    })

bayes_N_results = pd.DataFrame(all_results)

bayes_N_results

Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [alpha, tau, beta, gamma]


Output()

Sampling 4 chains for 1_000 tune and 1_000 draw iterations (4_000 + 4_000 draws total) took 55 seconds.
Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [alpha, tau, beta, gamma]


Output()

Sampling 4 chains for 1_000 tune and 1_000 draw iterations (4_000 + 4_000 draws total) took 46 seconds.
Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [alpha, tau, beta, gamma]


...

Output()

Sampling 4 chains for 1_000 tune and 1_000 draw iterations (4_000 + 4_000 draws total) took 94 seconds.
Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [alpha, tau, beta, gamma]


Output()

Sampling 4 chains for 1_000 tune and 1_000 draw iterations (4_000 + 4_000 draws total) took 1442 seconds.


,N,tau_mean,tau_CI_lower,tau_CI_upper,P_tau_positive,ATE_mean,ATE_CI_lower,ATE_CI_upper,P_ATE_above_0.01
0,200,0.361880,-0.634296,1.289359,0.76775,0.022791,-0.040056,0.084230,0.66450
1,500,0.411893,-0.198418,1.040700,0.91225,0.030770,-0.014725,0.078247,0.81075
2,1000,0.478302,-0.003323,0.949069,0.97475,0.030902,-0.000226,0.061610,0.90800
3,3000,0.430802,0.165753,0.693555,0.99900,0.032335,0.012386,0.052066,0.98700
4,6000,0.374119,0.186993,0.561225,0.99975,0.028198,0.014156,0.041855,0.99475
5,10000,0.311687,0.160640,0.458139,1.00000,0.022924,0.011919,0.033509,0.98850
6,200000,0.349413,0.317670,0.381033,1.00000,0.026383,0.023982,0.028758,1.00000


This analysis shows how increasing the sample size changes the Bayesian posterior under the same prior:

$$
\tau \sim N(0,1)
$$

The true treatment coefficient used to generate the data is:

$$
\tau_{\text{true}}=0.35
$$

## ATE on the probability scale

The Bayesian ATE is:

$$
ATE
=
\frac{1}{N}
\sum_{i=1}^{N}
\left[
p_i(D=1)-p_i(D=0)
\right]
$$



# Treatment Coefficient on the Log-Odds Scale

The true treatment coefficient is:

$$
\tau_{\text{true}}=0.35
$$

## \(N=200\)

$$
E(\tau\mid Y)=0.362
$$

$$
95\%\,CI=[-0.634,\;1.289]
$$

$$
P(\tau>0\mid Y)=76.8\%
$$

The posterior mean is close to the true value, but the interval is very wide and includes zero. There is only moderate evidence that the treatment effect is positive.

## \(N=500\)

$$
E(\tau\mid Y)=0.412
$$

$$
95\%\,CI=[-0.198,\;1.041]
$$

$$
P(\tau>0\mid Y)=91.2\%
$$

The interval becomes narrower, and the probability of a positive treatment effect increases. However, negative effects are still included in the credible interval.

## \(N=1000\)

$$
E(\tau\mid Y)=0.478
$$

$$
95\%\,CI=[-0.003,\;0.949]
$$

$$
P(\tau>0\mid Y)=97.5\%
$$

The posterior strongly favors a positive treatment effect. The lower bound is only slightly below zero, so the interval is almost entirely positive.

## \(N=3000\)

$$
E(\tau\mid Y)=0.431
$$

$$
95\%\,CI=[0.166,\;0.694]
$$

$$
P(\tau>0\mid Y)=99.9\%
$$

The credible interval is now entirely above zero. There is very strong evidence that the treatment effect is positive.

## \(N=6000\)

$$
E(\tau\mid Y)=0.374
$$

$$
95\%\,CI=[0.187,\;0.561]
$$

$$
P(\tau>0\mid Y)=99.98\%
$$

The posterior mean is very close to the true value of 0.35, and the credible interval is narrower than before.

## \(N=10000\)

$$
E(\tau\mid Y)=0.312
$$

$$
95\%\,CI=[0.161,\;0.458]
$$

$$
P(\tau>0\mid Y)=100\%
$$

The estimate moves slightly below the true value because of sampling variation, but the interval remains entirely positive and still contains 0.35.

## \(N=200000\)

$$
E(\tau\mid Y)=0.3494
$$

$$
95\%\,CI=[0.3177,\;0.3810]
$$

$$
P(\tau>0\mid Y)=100\%
$$

The posterior mean is almost exactly equal to the true treatment coefficient:

$$
0.3494\approx0.35
$$

The interval is very narrow, showing that the treatment coefficient is estimated with high precision.



### Main Conclusion

As the sample size increases:

$$
\text{credible-interval width}\downarrow
$$

$$
P(\tau>0\mid Y)\uparrow
$$

$$
P(ATE>0.01\mid Y)\uparrow
$$

Thus, larger samples reduce posterior uncertainty and provide stronger evidence that the treatment has a positive and practically meaningful effect.

For most managers, the Bayesian statement is easier to understand:

$$
P(\tau>0\mid\text{data})=97\%
$$

> “Given the model, prior, and observed data, there is a 97% posterior probability that the treatment effect is positive.”

This directly answers the business question:

> “How likely is it that the treatment works?”

The frequentist statement:

> “We reject the null hypothesis at the 5% significance level.”

is less intuitive. It means the observed result would be sufficiently unusual under the assumption of no treatment effect. It does not mean:

$$
P(\tau>0\mid\text{data})=95\%
$$

and it does not mean there is only a 5% probability that the null hypothesis is true.

For managerial decisions, Bayesian results can also express practical value:

$$
P(ATE>0.01\mid\text{data})=90.8\%
$$

> “There is a 90.8% probability that the treatment improves purchases by more than one percentage point.”

That is usually more useful than merely saying the effect is statistically significant. Therefore, Bayesian statements are generally more intuitive for managers, while frequentist statements remain common because they are standardized and widely used in scientific and regulatory reporting.